# Using `run_mc_powerflow_from_sqlite.py`

This notebook provides examples for running multiconductor powerflow from a SQLite database.

## Overview

The `run_mc_powerflow_from_sqlite.py` module provides functions to:
1. **Build a multiconductor network** from a SQLite database (`build_network_from_db`)
2. **Run powerflow** and write results back to the database (`run_powerflow`)

The SQLite database follows a multi-network schema where each network is identified by a `network_id`.

## Setup

In [1]:
import sys
from pathlib import Path

# Add the repo root to path if needed
repo_root = Path.cwd().parent if Path.cwd().name == "backend" else Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

# Import the module
from backend.run_mc_powerflow_from_sqlite import (
    build_network_from_db,
    run_powerflow,
)

import sqlite3
import pandas as pd

## 1. Explore Available Networks in the Database

Before running powerflow, you can inspect the available networks in the SQLite database.

In [2]:
# Path to the SQLite database
db_path = Path("multiconductor.db")

# Connect and list available networks
conn = sqlite3.connect(db_path)
networks_df = pd.read_sql_query("SELECT network_id, name, sn_mva, f_hz FROM networks", conn)
conn.close()

print(f"Available networks in {db_path}:")
display(networks_df)

Available networks in multiconductor.db:


,network_id,name,sn_mva,f_hz
0,CKT_114_16955,St. Charles,100.0,60.0


## 2. Build a Network from SQLite

Use `build_network_from_db()` to load a multiconductor network object from the database.
This reads all component tables (bus, line, ext_grid, asymmetric_load, etc.) and reconstructs the network.

In [3]:
# Select a network_id to load
network_id = networks_df["network_id"].iloc[0] if len(networks_df) > 0 else None
print(f"Loading network: {network_id}")

# Build the network
net = build_network_from_db(db_path, network_id=network_id)

print(f"\nNetwork name: {net.name}")
print(f"System frequency: {net.f_hz} Hz")
print(f"Base MVA: {net.sn_mva} MVA")

Loading network: CKT_114_16955

Network name: 
System frequency: 60.0 Hz
Base MVA: 100.0 MVA


## 3. Inspect Network Components

The network object contains DataFrames for each component type.

In [4]:
# List all component tables
print("Component tables in network:")
for key in net.keys():
    if isinstance(net[key], pd.DataFrame) and len(net[key]) > 0:
        print(f"  {key}: {len(net[key])} rows")

Component tables in network:
  bus: 1960 rows
  ext_grid_sequence: 3 rows
  asymmetric_load: 183 rows
  asymmetric_sgen: 166 rows
  line: 369 rows
  switch: 782 rows
  trafo1ph: 366 rows
  controller: 19 rows
  asymmetric_shunt: 15 rows
  res_bus: 1960 rows
  res_line: 369 rows
  res_trafo: 366 rows


In [5]:
# View buses
print("Buses:")
display(net.bus.head(10))

Buses:


network_id              name  vn_kv  grounded  \
index phase                                                     
0     0      CKT_114_16955   -99999942981806   0.24      True   
      1      CKT_114_16955   -99999942981806   0.24     False   
      2      CKT_114_16955   -99999942981806   0.24     False   
      3      CKT_114_16955   -99999942981806   0.24     False   
1     0      CKT_114_16955  -777777177282953  16.00      True   
      1      CKT_114_16955  -777777177282953  16.00     False   
      2      CKT_114_16955  -777777177282953  16.00     False   
      3      CKT_114_16955  -777777177282953  16.00     False   
2     0      CKT_114_16955          55346818  16.00      True   
      1      CKT_114_16955          55346818  16.00     False   

             grounding_r_ohm  grounding_x_ohm  in_service type  zone  
index phase                                                           
0     0                  0.0              0.0        True    b  None  
      1                  NaN              NaN        True    b  None  
      2                  NaN              NaN        True    b  None  
      3                  NaN              NaN        True    b  None  
1     0                  0.0              0.0        True    b  None  
      1                  NaN              NaN        True    b  None  
      2                  NaN              NaN        True    b  None  
      3                  NaN              NaN        True    b  None  
2     0                  0.0              0.0        True    b  None  
      1                  NaN              NaN        True    b  None

In [6]:
# View lines
print("Lines:")
display(net.line.head(10))

Lines:


network_id                   name                  std_type  \
index circuit                                                                   
0     0        CKT_114_16955       4382922E$508578H  6_CU_2PH_6_CU_N_16KV_3_N   
      1        CKT_114_16955       4382922E$508578H  6_CU_2PH_6_CU_N_16KV_3_N   
      2        CKT_114_16955       4382922E$508578H  6_CU_2PH_6_CU_N_16KV_3_N   
1     0        CKT_114_16955   ND206573398$4761140E       1/0_ACSR_2PH_16KV_2   
      1        CKT_114_16955   ND206573398$4761140E       1/0_ACSR_2PH_16KV_2   
2     0        CKT_114_16955  B5054768-2$B5054765-1          2_CLP_2PH_16KV_2   
      1        CKT_114_16955  B5054768-2$B5054765-1          2_CLP_2PH_16KV_2   
3     0        CKT_114_16955      4513975E$1387682E       1/0_ACSR_3PH_16KV_3   
      1        CKT_114_16955      4513975E$1387682E       1/0_ACSR_3PH_16KV_3   
      2        CKT_114_16955      4513975E$1387682E       1/0_ACSR_3PH_16KV_3   

              model_type  from_bus  from_phase  to_bus  to_phase  length_km  \
index circuit                                                                 
0     0           matrix       110           0      99         0   0.033675   
      1           matrix       110           1      99         1   0.033675   
      2           matrix       110           2      99         2   0.033675   
1     0           matrix       295           1     377         1   0.031908   
      1           matrix       295           2     377         2   0.031908   
2     0           matrix       243           1     470         1   0.062063   
      1           matrix       243           3     470         3   0.062063   
3     0           matrix       207           1     479         1   0.040496   
      1           matrix       207           2     479         2   0.040496   
      2           matrix       207           3     479         3   0.040496   

               type  in_service  
index circuit                    
0     0        None        True  
      1        None        True  
      2        None        True  
1     0        None        True  
      1        None        True  
2     0        None        True  
      1        None        True  
3     0        None        True  
      1        None        True  
      2        None        True

In [7]:
# View external grids (voltage sources)
print("External Grids:")
display(net.ext_grid.head(10))

External Grids:


,,network_id,name,bus,from_phase,to_phase,vm_pu,va_degree,r_ohm,x_ohm,in_service
index,circuit,,,,,,,,,,


In [8]:
# View loads
if hasattr(net, 'asymmetric_load') and len(net.asymmetric_load) > 0:
    print("Asymmetric Loads:")
    display(net.asymmetric_load.head(10))
else:
    print("No asymmetric loads in this network.")

Asymmetric Loads:


network_id      name  bus  from_phase  to_phase      p_mw  \
index circuit                                                                 
0     0        CKT_114_16955   774653H  280           1         2  0.000000   
      1        CKT_114_16955   774653H  280           2         3  0.000000   
      2        CKT_114_16955   774653H  280           3         1  0.000000   
1     0        CKT_114_16955   5054766  217           1         2  0.007490   
      1        CKT_114_16955   5054766  217           2         3  0.007490   
      2        CKT_114_16955   5054766  217           3         1  0.007490   
2     0        CKT_114_16955  1327372E  299           1         2  0.000815   
      1        CKT_114_16955  1327372E  299           2         3  0.000815   
      2        CKT_114_16955  1327372E  299           3         1  0.000815   
3     0        CKT_114_16955   227295E  195           2         0  0.000000   

                 q_mvar  const_z_percent_p  const_i_percent_p  \
index circuit                                                   
0     0        0.000000                0.0                0.0   
      1        0.000000                0.0                0.0   
      2        0.000000                0.0                0.0   
1     0        0.003946                0.0                0.0   
      1        0.003946                0.0                0.0   
      2        0.003946                0.0                0.0   
2     0        0.000355                0.0                0.0   
      1        0.000355                0.0                0.0   
      2        0.000355                0.0                0.0   
3     0        0.000000                0.0                0.0   

               const_z_percent_q  const_i_percent_q  sn_mva  scaling  \
index circuit                                                          
0     0                      0.0                0.0   0.025      1.0   
      1                      0.0                0.0   0.025      1.0   
      2                      0.0                0.0   0.025      1.0   
1     0                      0.0                0.0   0.100      1.0   
      1                      0.0                0.0   0.100      1.0   
      2                      0.0                0.0   0.100      1.0   
2     0                      0.0                0.0   0.025      1.0   
      1                      0.0                0.0   0.025      1.0   
      2                      0.0                0.0   0.025      1.0   
3     0                      0.0                0.0   0.050      1.0   

               in_service         type  
index circuit                           
0     0              True         None  
      1              True         None  
      2              True         None  
1     0              True   COMMERCIAL  
      1              True   COMMERCIAL  
      2              True   COMMERCIAL  
2     0              True  RESIDENTIAL  
      1              True  RESIDENTIAL  
      2              True  RESIDENTIAL  
3     0              True         None

## 4. Run Powerflow Manually

You can run the powerflow solver directly on the network object.

In [9]:
from multiconductor.pycci import cci_powerflow

# Run powerflow
try:
    cci_powerflow.run_pf(net)
    print("Powerflow converged successfully!")
except Exception as e:
    print(f"Powerflow failed: {e}")

Powerflow converged successfully!


## 5. Inspect Results

After running powerflow, results are stored in `res_*` attributes.

In [10]:
# Bus voltage results
print("Bus Results (res_bus):")
display(net.res_bus.head(10))

Bus Results (res_bus):


vm_pu   va_degree          p_mw        q_mvar  \
index phase                                                     
0     0           NaN         NaN -0.000000e+00 -0.000000e+00   
      1      1.011769   -0.793538  1.172068e-02  6.173128e-03   
      2      1.011890 -120.779244  1.172054e-02  6.176793e-03   
      3      1.012048  119.207665  1.172378e-02  6.175079e-03   
1     0           NaN         NaN -0.000000e+00 -0.000000e+00   
      1      1.039886   -0.481210 -0.000000e+00 -0.000000e+00   
      2      1.040004 -120.467456  3.067703e-13  2.482094e-13   
      3      1.040174  119.518136  2.518255e-13 -6.975155e-14   
2     0           NaN         NaN -0.000000e+00 -0.000000e+00   
      1      1.032283   -0.168460 -1.218844e-13 -2.475152e-12   

             imbalance_percent  
index phase                     
0     0               0.014403  
      1               0.014403  
      2               0.014403  
      3               0.014403  
1     0               0.014667  
      1               0.014667  
      2               0.014667  
      3               0.014667  
2     0               0.002389  
      1               0.002389

In [11]:
# Line results
print("Line Results (res_line):")
display(net.res_line.head(10))

Line Results (res_line):


i_from_ka  ia_from_degree   i_to_ka  ia_to_degree      i_ka  \
index circuit                                                                
0     0              NaN             NaN       NaN           NaN       NaN   
      1              NaN             NaN       NaN           NaN       NaN   
      2              NaN             NaN       NaN           NaN       NaN   
1     0         0.002618        5.932339  0.002618   -174.067661  0.002618   
      1         0.002618     -174.040512  0.002618      5.959488  0.002618   
2     0         0.001569      -58.371172  0.001569    121.628828  0.001569   
      1         0.001570      121.609769  0.001570    -58.390231  0.001570   
3     0         0.001121      -28.445222  0.001121    151.554778  0.001121   
      1         0.001121     -148.445210  0.001121     31.554790  0.001121   
      2         0.001121       91.554780  0.001121    -88.445220  0.001121   

               p_from_mw   p_to_mw  q_from_mvar  q_to_mvar         pl_mw  \
index circuit                                                              
0     0              NaN       NaN          NaN        NaN           NaN   
      1              NaN       NaN          NaN        NaN           NaN   
      2              NaN       NaN          NaN        NaN           NaN   
1     0         0.024949 -0.024949    -0.002735   0.002735  1.549287e-07   
      1         0.014843 -0.014843     0.020228  -0.020228  1.548641e-07   
2     0         0.007974 -0.007974     0.012757  -0.012757  1.696054e-07   
      1         0.015041 -0.015041    -0.000524   0.000524  1.696654e-07   
3     0         0.009497 -0.009497     0.005051  -0.005051  3.603878e-08   
      1         0.009497 -0.009497     0.005054  -0.005054  3.603878e-08   
      2         0.009500 -0.009500     0.005052  -0.005052  3.603879e-08   

                    ql_mvar  vm_from_pu  vm_to_pu  va_from_degree  \
index circuit                                                       
0     0                 NaN         NaN       NaN             NaN   
      1                 NaN    1.036929  1.036908       -0.349366   
      2                 NaN    1.037022  1.037005     -120.321344   
1     0        1.100484e-07    1.037781  1.037775       -0.322651   
      1        1.100025e-07    1.037627  1.037620     -120.310342   
2     0        2.612782e-08    1.037724  1.037716       -0.379702   
      1        2.613707e-08    1.037969  1.037957      119.614867   
3     0        2.559894e-08    1.038859  1.038855       -0.436519   
      1        2.559894e-08    1.039020  1.039016     -120.422400   
      2        2.559895e-08    1.039174  1.039170      119.560950   

               va_to_degree  loading_percent  
index circuit                                 
0     0                 NaN              0.0  
      1           -0.349749              0.0  
      2         -120.320581              0.0  
1     0           -0.322939              0.0  
      1         -120.310205              0.0  
2     0           -0.379207              0.0  
      1          119.614745              0.0  
3     0           -0.436550              0.0  
      1         -120.422430              0.0  
      2          119.560920              0.0

In [12]:
# External grid results (power injection from source)
print("External Grid Results (res_ext_grid):")
display(net.res_ext_grid)

External Grid Results (res_ext_grid):


,network_id,circuit,p_mw,q_mvar
index,,,,


In [13]:
# Summary statistics
print("\n=== Voltage Summary ===")
print(f"Min voltage: {net.res_bus['vm_pu'].min():.4f} pu")
print(f"Max voltage: {net.res_bus['vm_pu'].max():.4f} pu")
print(f"Mean voltage: {net.res_bus['vm_pu'].mean():.4f} pu")

if 'res_line' in net.keys() and len(net.res_line) > 0:
    print("\n=== Line Loading Summary ===")
    print(f"Max loading: {net.res_line['loading_percent'].max():.2f}%")
    print(f"Mean loading: {net.res_line['loading_percent'].mean():.2f}%")


=== Voltage Summary ===
Min voltage: 0.0000 pu
Max voltage: 1.0402 pu
Mean voltage: 0.9874 pu

=== Line Loading Summary ===
Max loading: 0.00%
Mean loading: 0.00%


## 6. Run Powerflow and Save Results to Database

Use `run_powerflow()` to run powerflow and automatically write results back to the SQLite database.

In [14]:
# Run powerflow and write results to DB
run_powerflow(db_path, network_id=network_id)
print(f"Powerflow complete! Results written to {db_path} for network_id={network_id}")

Powerflow complete! Results written to multiconductor.db for network_id=CKT_114_16955


## 7. Query Results Directly from the Database

After running powerflow, you can query results directly from the database.

In [15]:
conn = sqlite3.connect(db_path)

# Query bus voltage results
res_bus = pd.read_sql_query(
    "SELECT * FROM res_bus WHERE network_id = ?",
    conn,
    params=(network_id,)
)
print("Bus results from database:")
display(res_bus.head(10))

conn.close()

Bus results from database:


,network_id,index,phase,vm_pu,va_degree,p_mw,q_mvar,imbalance_percent
0,CKT_114_16955,0,0,NaN,NaN,0.000000e+00,0.000000e+00,0.014403
1,CKT_114_16955,0,1,1.011769,-0.793538,1.172068e-02,6.173128e-03,0.014403
2,CKT_114_16955,0,2,1.011890,-120.779244,1.172054e-02,6.176793e-03,0.014403
3,CKT_114_16955,0,3,1.012048,119.207665,1.172378e-02,6.175079e-03,0.014403
4,CKT_114_16955,1,0,NaN,NaN,0.000000e+00,0.000000e+00,0.014667
5,CKT_114_16955,1,1,1.039886,-0.481210,0.000000e+00,0.000000e+00,0.014667
6,CKT_114_16955,1,2,1.040004,-120.467456,3.067703e-13,2.482094e-13,0.014667
7,CKT_114_16955,1,3,1.040174,119.518136,2.518255e-13,-6.975155e-14,0.014667
8,CKT_114_16955,2,0,NaN,NaN,0.000000e+00,0.000000e+00,0.002389
9,CKT_114_16955,2,1,1.032283,-0.168460,-1.218844e-13,-2.475152e-12,0.002389


In [16]:
conn = sqlite3.connect(db_path)

# Check the converged status from network_meta
meta = pd.read_sql_query(
    "SELECT key, value FROM network_meta WHERE network_id = ?",
    conn,
    params=(network_id,)
)
print("Network metadata:")
display(meta)

conn.close()

Network metadata:


,key,value
0,OPF_converged,false
1,converged,true
2,f_hz,60
3,format_version,"""3.0.0"""
4,name,""""""
5,rho_ohmm,1e-10
6,sn_mva,100
7,std_types,"{""configuration"": {""OH1"": {""conductor_outer_di..."
8,user_pf_options,{}
9,version,"""3.1.2"""


## 8. Batch Processing Multiple Networks

You can iterate over multiple networks to run powerflow on all of them.

In [ ]:
conn = sqlite3.connect(db_path)
network_ids = pd.read_sql_query("SELECT network_id FROM networks", conn)["network_id"].tolist()
conn.close()

results = []
for nid in network_ids[:5]:  # Process first 5 networks as example
    try:
        run_powerflow(db_path, network_id=nid)
        results.append({"network_id": nid, "status": "converged"})
        print(f"✓ {nid} - converged")
    except Exception as e:
        results.append({"network_id": nid, "status": "failed", "error": str(e)})
        print(f"✗ {nid} - failed: {e}")

print("\nBatch processing complete.")
display(pd.DataFrame(results))

## 9. Command Line Usage

The module can also be run from the command line:

```bash
# Run powerflow on a specific network
python run_mc_powerflow_from_sqlite.py --db multiconductor.db --network-id CKT_114_16955

# Use default database and network
python run_mc_powerflow_from_sqlite.py
```

## Summary of Key Functions

| Function | Description |
|----------|-------------|
| `build_network_from_db(db_path, network_id)` | Load a multiconductor network from SQLite |
| `run_powerflow(db_path, network_id)` | Run powerflow and write results to SQLite |

### Database Schema

Key tables in the SQLite database:

- **networks**: Network metadata (name, sn_mva, f_hz, etc.)
- **network_meta**: Additional key-value metadata per network
- **bus**: Bus/node definitions with phases
- **line**: Line/cable elements
- **ext_grid**: External grid connections (voltage sources)
- **asymmetric_load**: Phase-specific loads
- **trafo1ph**: Single-phase transformers
- **res_bus**: Bus voltage results
- **res_line**: Line flow results
- **res_ext_grid**: External grid power injection results